# 06 — Multi-Sensor Fusion & 3D Object Tracking

> **참고:** Udacity SFND_3D_Object_Tracking, RadarCameraFusion 포트폴리오
> **언어:** C++ (Udacity), Python (포트폴리오)
> **목표:** Camera-LiDAR 융합 3D 추적 + TTC 추정

---

## 6-1. 전체 파이프라인



```
Camera Frames          LiDAR Point Clouds
      │                        │
      ▼                        ▼
YOLO v3 Detection      RANSAC + Clustering
2D Bounding Boxes        3D Bounding Boxes
      │                        │
      └──────────┬─────────────┘
                 ▼
        ROI Crop + Association
        (LiDAR pts ↔ 2D bbox)
                 │
                 ▼
        Feature Tracking
        (SIFT/ORB 매칭)
                 │
                 ▼
        TTC 추정
        (Camera + LiDAR)
                 │
                 ▼
        EKF/UKF 상태 추적
```




---

## 6-2. ROI: LiDAR ↔ 2D Bounding Box 연관



In [ ]:
import numpy as np
import cv2

def associate_lidar_to_bbox(pts_lidar: np.ndarray,   # (N, 4) x,y,z,i
                              bboxes: list,            # [{'box': [x1,y1,x2,y2], ...}]
                              R_L2C: np.ndarray,
                              t_L2C: np.ndarray,
                              K: np.ndarray,
                              min_x: float = 0.0,
                              shrink: float = 0.1) -> dict:
    """각 bbox 내에 포함된 LiDAR 포인트 인덱스 반환"""
    # LiDAR → 카메라 좌표
    pts_cam = (R_L2C @ pts_lidar[:, :3].T).T + t_L2C
    # 카메라 앞쪽 + 최소 거리 필터
    mask_front = pts_cam[:, 2] > min_x
    pts_cam = pts_cam[mask_front]
    orig_idx = np.where(mask_front)[0]

    # 이미지 투영
    uvw = (K @ pts_cam.T).T
    uv = (uvw[:, :2] / uvw[:, 2:3]).astype(int)

    result = {}
    for i, bbox_info in enumerate(bboxes):
        x1, y1, x2, y2 = bbox_info['box']
        # shrink ratio로 edge 노이즈 제거
        dx = int((x2 - x1) * shrink)
        dy = int((y2 - y1) * shrink)
        in_box = ((uv[:, 0] > x1 + dx) & (uv[:, 0] < x2 - dx) &
                  (uv[:, 1] > y1 + dy) & (uv[:, 1] < y2 - dy))
        result[i] = orig_idx[in_box]
    return result




---

## 6-3. TTC — LiDAR 기반



In [ ]:
def compute_ttc_lidar(pts_prev: np.ndarray, pts_curr: np.ndarray,
                       ego_vel: float, dt: float = 0.1) -> float:
    """
    전방 물체까지의 거리 변화율로 TTC 추정
    pts: (N, 3) LiDAR 포인트 (x = 전방 거리)

    TTC = d₁ / (d₀ - d₁) × dt
    """
    if len(pts_prev) == 0 or len(pts_curr) == 0:
        return float('inf')

    # outlier 제거: 중앙값 기준 ±1σ 이내
    def robust_min_x(pts, margin=0.1):
        x_vals = pts[:, 0]
        med = np.median(x_vals)
        std = np.std(x_vals)
        filtered = x_vals[(x_vals > med - std) & (x_vals < med + std)]
        return filtered.min() if len(filtered) > 0 else med

    d0 = robust_min_x(pts_prev)   # 이전 최근접 거리
    d1 = robust_min_x(pts_curr)   # 현재 최근접 거리

    if abs(d0 - d1) < 1e-3:
        return float('inf')

    ttc = d1 / (d0 - d1) * dt
    return max(ttc, 0.0)




---

## 6-4. 데이터 연관 (Hungarian Algorithm)

멀티 오브젝트 추적의 핵심: 예측 트랙 ↔ 현재 탐지 최적 매칭.



In [ ]:
from scipy.optimize import linear_sum_assignment
import numpy as np

def iou_2d(box1, box2) -> float:
    """2D bbox IoU [x1, y1, x2, y2]"""
    ix1 = max(box1[0], box2[0])
    iy1 = max(box1[1], box2[1])
    ix2 = min(box1[2], box2[2])
    iy2 = min(box1[3], box2[3])
    inter = max(0, ix2-ix1) * max(0, iy2-iy1)
    a1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    a2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = a1 + a2 - inter
    return inter / union if union > 0 else 0.0

def hungarian_associate(tracks: list, detections: list,
                         iou_threshold: float = 0.3):
    """
    tracks    : [{'box': ..., 'id': ...}, ...]
    detections: [{'box': ...}, ...]
    Returns: matched [(track_idx, det_idx)], unmatched_tracks, unmatched_dets
    """
    if not tracks or not detections:
        return [], list(range(len(tracks))), list(range(len(detections)))

    # 비용 행렬 (1 - IoU)
    cost = np.zeros((len(tracks), len(detections)))
    for i, t in enumerate(tracks):
        for j, d in enumerate(detections):
            cost[i, j] = 1.0 - iou_2d(t['box'], d['box'])

    row_idx, col_idx = linear_sum_assignment(cost)

    matched, unmatched_tracks, unmatched_dets = [], [], []
    det_matched = set()

    for r, c in zip(row_idx, col_idx):
        if cost[r, c] < 1.0 - iou_threshold:
            matched.append((r, c))
            det_matched.add(c)
        else:
            unmatched_tracks.append(r)

    unmatched_dets = [j for j in range(len(detections)) if j not in det_matched]
    return matched, unmatched_tracks, unmatched_dets




---

## 6-5. Track 생명주기 관리



In [ ]:
class Track:
    def __init__(self, detection: dict, track_id: int):
        self.id = track_id
        self.box = detection['box']
        self.lidar_pts = detection.get('lidar_pts', np.empty((0, 3)))
        self.hits = 1
        self.misses = 0
        self.state = 'tentative'  # tentative → confirmed → deleted
        self.kf = self._init_kf(detection)

    def _init_kf(self, det):
        # EKF 초기화 (5-state CTRV)
        x0 = np.array([det.get('cx', 0), det.get('cy', 0), 5.0, 0.0, 0.0])
        P0 = np.diag([4.0, 4.0, 25.0, 1.0, 0.1])
        return EKF_CTRV(x0, P0)

class MultiObjectTracker:
    def __init__(self, n_init=3, max_age=5, iou_thresh=0.3):
        self.tracks  = []
        self.next_id = 0
        self.n_init  = n_init     # confirmed 되기 위한 최소 hit 수
        self.max_age = max_age    # 삭제 전 최대 miss 수

    def update(self, detections: list, dt: float = 0.1):
        # 1. 예측
        for t in self.tracks:
            t.kf.predict(dt)

        # 2. 연관
        matched, unmatched_trk, unmatched_det = hungarian_associate(
            [{'box': t.box} for t in self.tracks],
            detections
        )

        # 3. 매칭된 트랙 업데이트
        for trk_idx, det_idx in matched:
            t = self.tracks[trk_idx]
            d = detections[det_idx]
            t.box = d['box']
            t.lidar_pts = d.get('lidar_pts', np.empty((0, 3)))
            t.hits  += 1
            t.misses = 0
            if t.hits >= self.n_init:
                t.state = 'confirmed'

        # 4. 미매칭 트랙 처리
        for trk_idx in unmatched_trk:
            t = self.tracks[trk_idx]
            t.misses += 1
            if t.misses >= self.max_age:
                t.state = 'deleted'

        # 5. 새 탐지 → 트랙 생성
        for det_idx in unmatched_det:
            new_track = Track(detections[det_idx], self.next_id)
            self.next_id += 1
            self.tracks.append(new_track)

        # 6. 삭제된 트랙 제거
        self.tracks = [t for t in self.tracks if t.state != 'deleted']

        return [t for t in self.tracks if t.state == 'confirmed']




---

## 6-6. Camera-Radar 순차 업데이트 (RadarCameraFusion 방식)



In [ ]:
class RadarCameraEKF:
    """
    상태: [cx, cy, vx, vy] (이미지 평면 + 속도)
    Camera: (cx, cy) 위치 업데이트
    Radar:  depth (Z), Doppler velocity 업데이트
    """
    def __init__(self, cx, cy, dt=0.5):
        self.dt = dt
        self.x  = np.array([cx, cy, 0., 0.])
        self.P  = np.diag([50., 50., 250., 250.])
        self.F  = np.array([[1,0,dt,0],[0,1,0,dt],[0,0,1,0],[0,0,0,1]])
        self.Q  = np.diag([1., 1., 10., 10.])

    def predict(self):
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q

    def update_camera(self, cx: float, cy: float):
        H = np.array([[1,0,0,0],[0,1,0,0]])
        R = np.diag([25., 25.])   # ~5px std
        z = np.array([cx, cy])
        y = z - H @ self.x
        S = H @ self.P @ H.T + R
        K = self.P @ H.T @ np.linalg.inv(S)
        self.x += K @ y
        self.P = (np.eye(4) - K @ H) @ self.P

    def update_radar_depth(self, depth: float, doppler: float,
                            K_cam: np.ndarray):
        """
        depth: 레이더 거리 (m) → 픽셀 위치 보정에 사용
        doppler: 레이더 시선 속도 (m/s) → KF vx 업데이트
        """
        # Doppler → 픽셀 속도 근사 (focal length 변환)
        fx = K_cam[0, 0]
        vx_meas = doppler * fx / max(depth, 1.0)
        H = np.array([[0,0,1,0]])
        R = np.array([[0.25]])   # ~0.5 m/s std
        y = np.array([vx_meas]) - H @ self.x
        S = H @ self.P @ H.T + R
        K = self.P @ H.T @ np.linalg.inv(S)
        self.x += K @ y
        self.P = (np.eye(4) - K @ H) @ self.P




---

## 6-7. C++ 구조 (Udacity SFND_3D_Object_Tracking)



```cpp
// camFusion_Student.cpp — clusterLidarWithROI
void clusterLidarWithROI(std::vector<BoundingBox>& boundingBoxes,
                          std::vector<LidarPoint>& lidarPoints,
                          float shrinkFactor, cv::Mat& P_rect,
                          cv::Mat& R_rect, cv::Mat& RT)
{
    for (auto it1 = lidarPoints.begin(); it1 != lidarPoints.end(); ++it1) {
        // LiDAR → 이미지 투영
        cv::Mat X(4,1,CV_64F);
        X.at<double>(0) = it1->x; X.at<double>(1) = it1->y;
        X.at<double>(2) = it1->z; X.at<double>(3) = 1;
        cv::Mat Y = P_rect * R_rect * RT * X;
        cv::Point pt(Y.at<double>(0,0)/Y.at<double>(2,0),
                     Y.at<double>(1,0)/Y.at<double>(2,0));

        for (auto it2 = boundingBoxes.begin(); it2 != boundingBoxes.end(); ++it2) {
            cv::Rect smallerBox = it2->roi;
            // shrink로 edge 노이즈 제거
            smallerBox.x     += shrinkFactor * smallerBox.width / 2.0;
            smallerBox.y     += shrinkFactor * smallerBox.height / 2.0;
            smallerBox.width  = smallerBox.width * (1 - shrinkFactor);
            smallerBox.height = smallerBox.height * (1 - shrinkFactor);
            if (smallerBox.contains(pt))
                it2->lidarPoints.push_back(*it1);
        }
    }
}
```




---

## 핵심 개념 정리

| 개념 | 핵심 |
|------|------|
| ROI Association | LiDAR 점 투영 → 2D bbox 내부 여부로 연관 |
| TTC LiDAR | 최근접 거리 변화율, 중앙값으로 outlier 강건화 |
| TTC Camera | 키포인트 간 거리 비율 변화, Lowe ratio 필터 |
| Hungarian | 비용 행렬 최적 할당 O(n³), IoU 기반 |
| Track lifecycle | tentative→confirmed(n_init hits), max_age misses로 삭제 |

---

## 다음 단계

→ **07_advanced.md** : JPDA, GM-PHD, PMBM, PointPainting, 딥러닝 융합
